# Data Cleaning: Impacts of Data Centers on Electrical Utilities

This notebook loads the raw data that `01-data_collection.ipynb` saved to `data/raw/` and cleans it.

The main output is the cleaned FracTracker data centers table with an `is_crypto` label.

The EIA sections near the end are a first look only. They are not saved yet.

In [182]:
import re
import pandas as pd
import numpy as np
from pathlib import Path
import geocoder
from geopy.distance import geodesic

In [183]:
DATA_DIR = Path("data/raw")

# display options for pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

## First look at each dataset

### FracTracker U.S. Data Centers

In [184]:
data_centers = pd.read_csv(DATA_DIR / "fractracker_data_centers.csv")
print("Rows and columns:", data_centers.shape)
data_centers.sample(3)

Rows and columns: (1708, 53)


,OBJECTID,GlobalID,facility_id,facility_name,address,city,state,zip,county,lat,long,status,location_confidence,location_determination,purpose,operator_name,tenant,mw,mw_low,mw_high,sizerank,sizerank_numeric,power_source,dedicated_power_plant,number_of_generators,number_of_buildings,cooling_source,cooling_type,facility_size_sqft,property_size_acres,project_cost,expected_date_online,community_pushback,advocacy_information,resistance_status,nda,community_group_website_1,community_group_website_2,petition_url,other_info,editor,information_source,info_source_1,info_source_2,info_source_3,info_source_4,info_source_5,info_source_6,info_source_7,info_source_8,date_created,date_updated,delete_dup
568,3952355,ac4b3538-4b85-4808-b5c4-6a5f7aecfae8,1767328905564-6832,Amazon Madison Mega Site,1600 West Peace St,Canton,MS,39046.0,Madison,32.595337,-90.096180,Operating,High,Google Maps and other sources,NaN,Amazon,NaN,341-819,341.0,819.0,Hyperscale (100-999 MW),5,NaN,NaN,NaN,NaN,Air,Fans,NaN,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KE,Media Monitoring,https://epoch.ai/data/data-centers/satellite-e...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.767226e+12,1.767226e+12,NaN
254,3952041,9822ddcd-904c-4160-9e1c-db46955043fd,163,DC BLOX Atlanta West Campus,1701 North River Road,Lithia Springs,GA,30122.0,Douglas,33.733944,-84.615234,Operating,High,Other sources,NaN,DC BLOX,NaN,120,120.0,120.0,Hyperscale (100-999 MW),5,NaN,NaN,NaN,NaN,NaN,NaN,1250000.0,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Other,Sci4GA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.760573e+12,1.760573e+12,NaN
101,3951888,1525c444-a56f-4a0b-8a5d-537e3c0ed15a,1789747387448-4731,Vantage Aurora campus (Trident / CO21),E-470 and E. 48th Avenue (adjacent to N. Tibet...,Aurora,CO,80206.0,Adams,39.791900,-104.702400,Proposed,Medium,Google Maps only,NaN,Vantage Data Centers,NaN,335-500,334.0,500.0,Hyperscale (100-999 MW),5,Grid (unspecified mix),NaN,NaN,2,Water,Closed loop,860000.0,108.0,NaN,NaN,Yes,NaN,Emerging Concern,NaN,NaN,NaN,https://www.change.org/p/protect-neighbourhood...,NaN,MG,Media Monitoring,https://www.planetizen.com/news/2026/09/138432...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.789690e+12,1.789690e+12,NaN


In [185]:
# Summary Statistics for Data Centers Dataset
# Looking for anything that needs to be cleaned
data_centers.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
OBJECTID,1708.0,NaN,NaN,NaN,3952640.5,493.201446,3951787.0,3952213.75,3952640.5,3953067.25,3953494.0
GlobalID,1708,1708,bfba4188-dbe1-42b9-9175-5b1c3011be98,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
facility_id,1708,1708,1779305646902-5353,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
facility_name,1708,1581,H5 Data Center,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
address,1663,1651,Farm-to-Market 157,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,1704,840,Sterling,82,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state,1708,49,VA,464,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zip,1695.0,NaN,NaN,NaN,43920.00944,26494.883341,1085.0,20166.0,30346.0,72815.5,99734.0
county,1705,480,Loudoun,192,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lat,1708.0,NaN,NaN,NaN,37.604335,4.226109,21.480452,33.784261,38.747388,40.058572,70.18478


### FracTracker U.S. Data Centers - Data Cleaning

| Feature | Identified Issue | Proposed Solution |
|---|---|---|
| `sizerank` / `sizerank_numeric` | Mismatched values that are not consistent across categories. | Align sizerank wiht sizerank_numeric column labels. |
| `mw` | Contains both numerical values and ranges. | Separate ranges into `mw_low` and `mw_high` to preserve the available information. |
| `power_source` | String categorical values. | Standardize categories and encode them for modeling. |
| `dedicated_power_plant` | Limited availability across facilities. | Determine if power plants can be identified using location (latitude/longitude) and power source. |
| `number_of_generators` | Contains numerical information stored as strings; limited availability across facilities. | Too limited to be a learnable feature unless more information can be added. |
| `number_of_buildings` | Contains integers, ranges, and strings (e.g., `6+`, `1-19`, `8`, `3 large, 1 small`). | Create a consistent approach for representing ranges and descriptive counts. |
| `cooling_source` | Categorical values stored as strings. | Standardize categories and encode them for modeling. |
| `cooling_type` | Inconsistent categorical formatting (e.g., `Closed loop` vs. `Closed Loop`). | Normalize categorical values and encode them for modeling. |
| `project_cost` | Contains strings and potentially ranges rather than consistent numerical values. | Extract numerical values and determine an appropriate method for representing ranges. |



`sizerank` / `sizerank_numeric`

In [ ]:
# Mappings
size_rank_mw = {
   'Unknown': 1,
   'Small (0-10 MW)': 2,
   'Medium (11-50 MW)': 3,
   'Large (51-99 MW)': 4,
   'Hyperscale (100-999 MW)': 5,
   'Mega campus (>1,000 MW)': 6
}

# Fix sizerank labels
data_centers['sizerank'] = data_centers['sizerank'].replace('Mega Campus (>1,000 MW)', 'Mega campus (>1,000 MW)')
data_centers['sizerank_numeric'] = data_centers['sizerank'].apply(lambda x: size_rank_mw[x])

`location_confidence`

In [186]:
# Small fix for location confidence values
data_centers['location_confidence'] = (
    data_centers['location_confidence']
    .apply(lambda x: 'High' if x == 'high' else x)
)

`sizerank`

In [187]:
# The Mega campus category is spelled two ways
data_centers["sizerank"] = data_centers["sizerank"].replace(
    "Mega Campus (>1,000 MW)", "Mega campus (>1,000 MW)"
)

data_centers["sizerank"].value_counts(dropna=False)

sizerank
Unknown                    888
Hyperscale (100-999 MW)    401
Medium (11-50 MW)          140
Mega campus (>1,000 MW)    122
Small (0-10 MW)            116
Large (51-99 MW)            41
Name: count, dtype: int64

`mw`

In [188]:
# Remove commas and spaces
mw = data_centers['mw'].str.replace(',', '').str.strip()

# Split and assign ranges
data_centers[['mw_low', 'mw_high']] = mw.str.split('-', expand=True)
data_centers['mw_high'] = data_centers['mw_high'].fillna(data_centers['mw_low']) # fill where no range exists 

# Convert to numeric
data_centers[['mw_low', 'mw_high']] = (
    data_centers[['mw_low', 'mw_high']]
    .apply(pd.to_numeric, errors='coerce')
)

data_centers.loc[data_centers['mw'].notna()][['mw', 'mw_low', 'mw_high']].head()

,mw,mw_low,mw_high
0,3000,3000.0,3000.0
1,120,120.0,120.0
2,1200,1200.0,1200.0
3,300,300.0,300.0
6,210,210.0,210.0


`power_source`

In [189]:
# Normalize power sources
power_source = (
    data_centers['power_source']
    .str.lower()
    .str.strip()
    .str.replace('(', '')
    .str.replace(')', '')
    .str.replace('/', '_')
    .str.replace(' ', '_')
)

# One-hot encoding for each source type
source_encoded = power_source.str.get_dummies(sep=",_")
data_centers = pd.concat([data_centers, source_encoded], axis=1)

columns = [
    "power_source", "batteries", "biomass", "coal", "fuel_cells", 
    "geothermal", "grid_unspecified_mix", "hybrid_onsite_and_grid", 
    "hydroelectric", "natural_gas", "nuclear", "oil_diesel", 
    "renewable", "solar", "wind"
]

data_centers.loc[data_centers['power_source'].notna()][columns].sample(5)

,power_source,batteries,biomass,coal,fuel_cells,geothermal,grid_unspecified_mix,hybrid_onsite_and_grid,hydroelectric,natural_gas,nuclear,oil_diesel,renewable,solar,wind
738,Natural gas,0,0,0,0,0,0,0,0,1,0,0,0,0,0
1146,Grid (unspecified mix),0,0,0,0,0,1,0,0,0,0,0,0,0,0
765,Natural gas,0,0,0,0,0,0,0,0,1,0,0,0,0,0
753,Hybrid (onsite and grid),0,0,0,0,0,0,1,0,0,0,0,0,0,0
1035,Solar,0,0,0,0,0,0,0,0,0,0,0,0,1,0


`number_of_buildings`

In [190]:
# Standardization function for assigning number of buildings
def standardize_buildings(buildings):
    if pd.isna(buildings):
        return np.nan, np.nan
    
    buildings = buildings.replace(' ', '')
    b_low = np.nan
    b_high = np.nan
    
    if '-' in buildings:
        split = buildings.split('-')
        b_low = int(split[0])
        b_high = int(split[1])
        
    elif '+' in buildings:
        buildings = buildings.replace('+', '')
        b_low = int(buildings)
        
    elif buildings.isdigit():
        b_low = int(buildings)
        b_high = int(buildings)
        
    return b_low, b_high
    
# Add new features to handle ranges
data_centers[["buildings_low", "buildings_high"]] = pd.DataFrame(
    data_centers["number_of_buildings"].apply(standardize_buildings).tolist(),
    index=data_centers.index
)

data_centers.loc[data_centers["buildings_low"].notna()][["number_of_buildings", "buildings_low", "buildings_high"]].head()

,number_of_buildings,buildings_low,buildings_high
2,18,18.0,18.0
10,4,4.0,4.0
11,2,2.0,2.0
14,14,14.0,14.0
15,6,6.0,6.0


`cooling_source` + `cooling_type`

In [191]:
for col in ['cooling_source', 'cooling_type']:
    # Normalize cooling cols
    cooling_data = (
        data_centers[col]
        .str.lower()
        .str.strip()
        .str.replace('/', '_')
        .str.replace(' ', '_')
    )

    # One-hot encoding for each source type
    source_encoded = cooling_data.str.get_dummies()
    data_centers = pd.concat([data_centers, source_encoded], axis=1)

columns = ['cooling_source', 'air', 'water', 'hybrid_air_water']

data_centers.loc[data_centers['cooling_source'].notna()][columns].sample(5)

,cooling_source,air,water,hybrid_air_water
812,Water,0,1,0
568,Air,1,0,0
340,Air,1,0,0
830,Air,1,0,0
421,Air,1,0,0


In [192]:
columns = ['cooling_type', 'closed_loop', 'fans', 'open_loop']

data_centers.loc[data_centers['cooling_type'].notna()][columns].sample(5)

,cooling_type,closed_loop,fans,open_loop
526,Open loop,0,0,1
125,Closed loop,1,0,0
377,Closed loop,1,0,0
575,Closed loop,1,0,0
692,Closed loop,1,0,0


`project_cost`

In [193]:
def standardize_project_cost(cost):
    if pd.isna(cost):
        return np.nan, np.nan
    
    multiplier = 1
    cost_low = np.nan
    cost_high = np.nan
    
    # Normalize cost
    cost = (
        str(cost)
        .lower()
        .replace(' ', '')
        .replace('$', '')
        .replace(',', '')
        .replace('miillion', 'million')
    )

    # Replace abbreviated m
    cost = re.sub(r'm$', 'million', cost)
    
    if 'million' in cost:
        multiplier = 1000000
    elif 'billion' in cost:
        multiplier = 1000000000
        
    cost = re.sub(r'[mb]illion', '', cost)
    
    # Assign range to low and high
    if '-' in cost:
        split = cost.split('-')
        cost_low = float(split[0]) * multiplier
        cost_high = float(split[1]) * multiplier
        
    elif '+' in cost:
        cost = cost.replace('+', '')
        cost_low = float(cost) * multiplier
        
    else:
        cost_low = float(cost) * multiplier
        cost_high = float(cost) * multiplier
        
    return cost_low, cost_high

# Add new features for handling the cost range
data_centers[["project_cost_low", "project_cost_high"]] = pd.DataFrame(
    data_centers["project_cost"].apply(standardize_project_cost).tolist(),
    index=data_centers.index
)

data_centers[data_centers["project_cost"].notna()][["project_cost", "project_cost_low", "project_cost_high"]].sample(5)

,project_cost,project_cost_low,project_cost_high
503,$1 billion,1.000000e+09,1.000000e+09
617,$3 billion,3.000000e+09,3.000000e+09
831,$26-$44 billion,2.600000e+10,4.400000e+10
15,$8 billion,8.000000e+09,8.000000e+09
966,$3 billion,3.000000e+09,3.000000e+09


### FracTracker Crypto Mining Operations

In [194]:
crypto = pd.read_csv(DATA_DIR / "fractracker_crypto.csv")
print("Rows and columns:", crypto.shape)
crypto.sample(3)

Rows and columns: (302, 18)


,Name,City,State,Address,Company,Probable power source,Links,Status,lat,long,Precise Location?,MapCat,GEM_links,Other updates,Date,MW (NYTimes),Unnamed: 16,Unnamed: 17
299,Mine under construction,Gillette,WY,"569 Breene Rd, Gillette, WY 82718",Unknown,NaN,https://www.sundancetimes.com/story/2023/06/15...,Proposed,43.854239,-105.385015,Yes,Operating / under construction,NaN,NaN,7/27/2023,NaN,NaN,NaN
153,Greenidge Operating,Underwood,ND,NaN,Greenidge,NaN,https://greenidge.com/our-operations/\nhttps:/...,Operating,47.455674,-101.134914,No,Operating / under construction,NaN,NaN,8/3/2024,7.5 MW,NaN,NaN
256,Fort Worth Mine,Fort Worth,TX,NaN,Geosyn Mining > Geosyntech Holdings,Unknown,https://www.send2press.com/wire/geosyn-mining-...,Operating,32.750136,-97.334205,No,Operating / under construction,NaN,NaN,NaN,NaN,NaN,NaN


In [195]:
crypto.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Name,302,250,CleanSpark,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,283,235,Dalton,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
State,302,41,GA,38,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Address,194,189,"430 Davis Rd, Vidalia, GA 30474",4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Company,283,174,CleanSpark,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Probable power source,163,36,Unknown,54,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Links,299,263,https://www.wbir.com/article/tech/east-tenness...,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Status,301,11,Operating,138,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lat,302.0,NaN,NaN,NaN,37.830255,5.191345,25.454595,34.456008,36.356141,41.261212,69.71761
long,302.0,NaN,NaN,NaN,-128.525789,651.005258,-11402.57827,-98.202512,-88.774187,-82.477328,-69.170992


### FracTracker Crypto Mining Operations - Data Cleaning

| Feature | Identified Issue | Proposed Solution |
|---|---|---|
| `Probable power source` | Power source values use inconsistent naming conventions. | Normalize them to the FracTracker data center categories using keyword mapping. |
| `Precise Location?` | Values represented as 'Yes' / 'No' strings. | Update to a boolean representation. |
| `MW (NYTimes)` | A range of values including '100 MW', '100 MW+', '100 MW --> 200 MW'. | Standardize to `mw_low` and `mw_high` to keep consistent representation in both datasets. |

`Probable power source`

In [196]:
# Reuse power sources from FracTracker data centers
power_sources = [
    "batteries", "biomass", "coal", "fuel_cells",
    "geothermal", "grid_unspecified_mix", "hybrid_onsite_and_grid",
    "hydroelectric", "natural_gas", "nuclear", "oil_diesel",
    "renewable", "solar", "wind"
]

# Mapping given sources to predefined power sources
def normalize_sources(source):
    if pd.isna(source):
        return []

    source = source.lower()
    found_sources = set()

    # Keyword mappings
    keyword_map = {
        "battery": "batteries",
        "biomass": "biomass",
        "coal": "coal",
        "fuel cell": "fuel_cells",
        "geothermal": "geothermal",
        "hydro": "hydroelectric",
        "gas": "natural_gas",
        "nuclear": "nuclear",
        "diesel": "oil_diesel",
        "oil": "oil_diesel",
        "solar": "solar",
        "wind": "wind",
        "renewable": "renewable",
        "wte": "renewable",
    }

    for keyword, category in keyword_map.items():
        if keyword in source:
            found_sources.add(category)

    # Grid
    if "grid" in source:
        found_sources.add("grid_unspecified_mix")

    # Grid + another identified source = hybrid
    non_grid_sources = found_sources - {"grid_unspecified_mix"}

    if "grid_unspecified_mix" in found_sources and non_grid_sources:
        found_sources.add("hybrid_onsite_and_grid")

    return sorted(found_sources)


crypto["norm_power_source"] = crypto["Probable power source"].apply(normalize_sources)
crypto.loc[crypto["norm_power_source"].str.len() > 0][["norm_power_source"]].sample(5)

,norm_power_source
215,[natural_gas]
65,[grid_unspecified_mix]
285,[grid_unspecified_mix]
204,[natural_gas]
286,[coal]


`Precise Location?`

In [197]:
# Change precise location to a boolean
crypto["Precise Location?"] = crypto["Precise Location?"].apply(lambda x: x == 'Yes')
crypto[['Precise Location?']].sample(5)

,Precise Location?
108,True
168,True
186,True
214,True
83,False


`MW (NYTimes)`

In [198]:
# Function for standardizing mw values
def standardize_mw(value):
    if pd.isna(value):
        return np.nan, np.nan

    text = str(value).lower().strip()

    # Fix typo
    text = text.replace("nw", "mw")

    # Extract numbers followed by MW or GW
    matches = re.findall(r'(\d+(?:\.\d+)?)\s*(mw|gw)', text)

    if not matches:
        return np.nan, np.nan

    values_mw = []

    for number, unit in matches:
        number = float(number)

        # Convert to MW
        if unit == "gw":
            number *= 1000

        values_mw.append(number)

    return min(values_mw), max(values_mw)


# Add new features to handle ranges
crypto[["mw_low", "mw_high"]] = pd.DataFrame(
    crypto["MW (NYTimes)"].apply(standardize_mw).tolist(),
    index=crypto.index
)

crypto.loc[crypto['MW (NYTimes)'].notna()][["MW (NYTimes)", "mw_low", "mw_high"]].sample(5)

,MW (NYTimes),mw_low,mw_high
62,275 MW,275.0,275.0
248,700 MW,700.0,700.0
147,92 MW,92.0,92.0
94,250 MW,250.0,250.0
109,13 MW,13.0,13.0


### Address Standardization & Comparison 

Addresses are standardized across both datasets to make facility matching more reliable. Common street names, directions, and unit types are converted to consistent abbreviations, while special characters and extra whitespace are removed. The crypto dataset sometimes stores the city, state, and ZIP code inside the Address field. These values are separated so the final location fields are: 

- **Crypto**: `Address`, `City`, `State`, `Zip` 
- **Data Centers**: `address`, `city`, `state`, `zip`

#### Address Abbreviations

In [199]:
ADDRESS_ABBREVIATIONS = {
    # Directions
    'north': 'n',
    'south': 's',
    'east': 'e',
    'west': 'w',
    'northeast': 'ne',
    'northwest': 'nw',
    'southeast': 'se',
    'southwest': 'sw',

    # Street suffixes
    'street': 'st',
    'avenue': 'ave',
    'road': 'rd',
    'drive': 'dr',
    'lane': 'ln',
    'boulevard': 'blvd',
    'circle': 'cir',
    'court': 'ct',
    'place': 'pl',
    'parkway': 'pkwy',
    'highway': 'hwy',
    'trail': 'trl',
    'terrace': 'ter',
    'square': 'sq',
    'crossing': 'xing',
    'center': 'ctr',
    'centre': 'ctr',
    'point': 'pt',
    'ridge': 'rdg',
    'run': 'run',
    'view': 'vw',
    'village': 'vlg',
    'station': 'sta',
    'landing': 'lndg',
    'junction': 'jct',
    'extension': 'ext',
    'expressway': 'expy',
    'freeway': 'fwy',
    'turnpike': 'tpke',
    'causeway': 'cswy',
    'crescent': 'cres',
    'cove': 'cv',
    'fork': 'frk',
    'garden': 'gdn',
    'gardens': 'gdns',
    'grove': 'grv',
    'harbor': 'hbr',
    'harbour': 'hbr',
    'heights': 'hts',
    'hill': 'hl',
    'hills': 'hls',
    'island': 'is',
    'islands': 'iss',
    'knoll': 'knl',
    'knolls': 'knls',
    'lake': 'lk',
    'lakes': 'lks',
    'manor': 'mnr',
    'manors': 'mnrs',
    'meadow': 'mdw',
    'meadows': 'mdws',
    'mount': 'mt',
    'mountain': 'mtn',
    'orchard': 'orch',
    'pine': 'pne',
    'pines': 'pnes',
    'plaza': 'plz',
    'port': 'prt',
    'prairie': 'pr',
    'radial': 'radl',
    'river': 'riv',
    'shore': 'shr',
    'shores': 'shrs',
    'spring': 'spg',
    'springs': 'spgs',
    'summit': 'smt',
    'union': 'un',
    'valley': 'vly',
    'valleys': 'vlys',
    'well': 'wl',
    'wells': 'wls',
    
    # Unit Overrides
    'apartment': 'apt',
    'apt.': 'apt',
    'suite': 'ste',
    'ste.': 'ste',
    'unit': 'unit',
    '#': 'unit',
    'building': 'bldg',
    'floor': 'fl',
    'room': 'rm',
    'department': 'dept'
}

#### Helper Functions

These functions are used to standardize addresses by removing special characters and extra whitespace and replacing common address terms with predefined abbreviations. In the crypto dataset, the `Address` field sometimes also contains the city, state, and ZIP code. A second function, `remove_city_state`, removes this duplicated location information, while ZIP codes are extracted into a separate column.

In [200]:
def standardize_address(address):
    if pd.isna(address):
        return np.nan
    
    address = address.lower().strip()
    
    # Remove special characters
    address = re.sub(r'[^a-z0-9\s]', '', address)
    address_parts = address.split()
    
    # Update addresses using abbreviations
    cleaned_parts = [
        ADDRESS_ABBREVIATIONS.get(part, part)
        for part in address_parts
    ]
    
    return " ".join(cleaned_parts)


def remove_city_state(address, city, state):
    if pd.isna(address):
        return np.nan

    address = str(address)

    # Remove ZIP from address
    address = re.sub(r'\s*,?\s*\d{5}\s*$', '', address)

    # Remove State from address
    if pd.notna(state):
        address = re.sub(
            rf'\s*,?\s*{re.escape(str(state))}\s*$',
            '',
            address,
            flags=re.IGNORECASE
        )

    # Remove City from address
    if pd.notna(city):
        address = re.sub(
            rf'\s*,?\s*{re.escape(str(city))}\s*$',
            '',
            address,
            flags=re.IGNORECASE
        )

    return address.strip(' ,')

In [201]:
# Extract 5-digit zip if exists
crypto['Zip'] = crypto['Address'].str.extract(r'(\d{5})\s*$')[0]
crypto['Zip'] = crypto['Zip'].replace('', np.nan).astype('Int64') # missing zip

# Remove city, state, and zip from addresses in crypto
crypto['Address'] = crypto.apply(
    lambda row: remove_city_state(
        row['Address'],
        row['City'],
        row['State']
    ),
    axis=1
)

# Applying the transformations to the addresses in both datasets
crypto['Address'] = crypto['Address'].apply(standardize_address)
data_centers['address'] = data_centers['address'].apply(standardize_address)


# data_centers[['address', 'city', 'state', 'zip']].sample(5)
crypto[['Address', 'City', 'State', 'Zip']].sample(5)

,Address,City,State,Zip
50,2380 godby rd college park georgia,College Park,GA,30349
217,NaN,New Tazewell,TN,<NA>
120,NaN,Minneapolis,MN,<NA>
4,grayhawk rd,Cabot,AR,72023
9,700 midway rd,Hoxie,AR,72433


#### Address Comparison

The standardized datasets are merged using exact matches on address, city, and state to identify facilities that appear in both datasets. This allows us to identify data centers associated with known crypto mining locations, providing more information for later analysis. Using exact address matching, 19 matching records were identified between the two datasets.

In [202]:
# Find matching addresses between the datasets
matches = crypto[crypto['Address'].notna()].merge(
    data_centers[data_centers['address'].notna()],
    left_on=['Address', 'City', 'State'],
    right_on=['address', 'city', 'state'],
    how='inner',
    suffixes=('_crypto', '_dc')
)

print(f"{len(matches)} matches found between the crypto and data centers datasets.")
matches[['facility_name', 'Name', 'lat_crypto', 'long_crypto', 'lat_dc', 'long_dc', 'address', 'city', 'state']]

19 matches found between the crypto and data centers datasets.


,facility_name,Name,lat_crypto,long_crypto,lat_dc,long_dc,address,city,state
0,Core Scientific Dalton 1 & 2 Data Center Campus,Core Scientific Whitfield County facility (sel...,34.713160,-84.951430,34.713160,-84.951430,2205 industrial s rd,Dalton,GA
1,Riot 1,Block Mining McCracken County facility,37.088310,-88.703050,37.088310,-88.703050,5600 commerce dr,Paducah,KY
2,Riot 2,Riot Platforms greenfield site,37.088592,-88.676158,37.088593,-88.676155,coleman rd,Paducah,KY
3,Hyperscale Data Michigan Data Center,Michigan Data Center,41.993228,-86.098603,41.992455,-86.098976,415 e pr ronde st,Dowagiac,MI
4,CoreWeave/ Applied Digital,Applied Digital Stutsman County facility,47.027383,-98.679666,47.027080,-98.679600,2671 hwy 20 se,Jamestown,ND
5,Applied Digital: Polaris Forge 1,"Applied Digital, being repurposed from crypto ...",46.010438,-98.570800,46.012590,-98.570700,9685 87th ave se,Ellendale,ND
6,Vulcan Infrastructure and Power Inc (formerly ...,Greenidge Generation Yates County facility,42.676870,-76.944910,42.676870,-76.944910,590 plant rd,Dresden,NY
7,BlockFusion aka North East Data Center LLC,North East Data LLC Niagara facility,43.082864,-79.003495,43.083187,-79.003170,5380 frontier ave,Niagara Falls,NY
8,9Lives Data Center,Nine Lives Digital Group,43.123208,-79.040614,43.123890,-79.041180,3807 highland ave,Niagara Falls,NY
9,North Tonawanda Data Center,Digipower X Fortistar Niagara County facility,43.060860,-78.853760,43.060860,-78.853760,1070 erie ave,North Tonawanda,NY


#### Crypto Label for Data Centers

The FracTracker data center list does not say which facilities are crypto mines. We build an `is_crypto` label here so `03-supervised-learning.ipynb` can load it from `data/processed/`.

The label comes from **two different sources**:

1. **The FracTracker crypto dataset.** A data center is crypto if it matches a known crypto mining site, either:
   - by location: a crypto site is within `MAX_MILES` of it, or
   - by exact address (the Address Comparison above).
2. **The FracTracker data centers dataset itself.** Some data centers already say they are crypto in their `purpose` or `facility_name` (for example "Bitcoin" or "Crypto mining/AI"). Many of these are not in the crypto dataset at all.

We keep one column for each check (`crypto_from_distance`, `crypto_from_address`, `crypto_from_dc_dataset`) so we can always see where a label came from.

##### How old is each dataset?

The crypto dataset is older than the data centers dataset:

- **Crypto sites:** the `Date` column runs from March 2023 to April 2026, but most entries are from 2023 and 2024 (median: May 2024).
- **Data centers:** the `date_updated` column runs from January 2025 to September 2026 (median: November 2025).

So there is roughly a year and a half between the typical crypto entry and the typical data center entry. This matters for the crypto label:

- A data center that matched a crypto site tells us the site **was** crypto when the crypto dataset was made. It may not still be crypto. Several sites are converting from crypto mining to AI (for example Applied Digital, Crexi and Keel Infrastructure).
- A "Proposed" crypto site from 2023 may never have been built, or may have been built as a different kind of data center.
- The `crypto_from_dc_dataset` column comes from the newer dataset, so it is the more up-to-date of the two sources.

In [203]:
# Compare how old each dataset is
crypto_dates = pd.to_datetime(crypto["Date"], format="%m/%d/%Y", errors="coerce")
dc_dates = pd.to_datetime(data_centers["date_updated"], unit="ms")

print("Crypto sites with a date:", crypto_dates.notna().sum(), "of", len(crypto))
print("Crypto dates:            ", crypto_dates.min().date(), "to", crypto_dates.max().date())
print("Crypto median date:      ", crypto_dates.median().date())
print()
print("Data centers last updated:", dc_dates.min().date(), "to", dc_dates.max().date())
print("Data centers median date: ", dc_dates.median().date())
print()
print("Crypto sites by year:")
print(crypto_dates.dt.year.value_counts().sort_index())
print()
print("Data centers by year last updated:")
print(dc_dates.dt.year.value_counts().sort_index())

Crypto sites with a date: 167 of 302
Crypto dates:             2023-03-02 to 2026-04-15
Crypto median date:       2024-05-17

Data centers last updated: 2025-01-04 to 2026-10-07
Data centers median date:  2025-11-24

Crypto sites by year:
Date
2023.0    39
2024.0    96
2025.0    30
2026.0     2
Name: count, dtype: int64

Data centers by year last updated:
date_updated
2025.0    893
2026.0    812
Name: count, dtype: int64


##### Drop invalid coordinates

A few rows have coordinates that cannot be real: a longitude of `-11402.57827` (likely a typo) and a data center at `0, 0` (missing location). We can't measure distance for these, so we drop them.

In [204]:
def valid_coordinates(df):
    return df["lat"].between(-90, 90) & df["long"].between(-180, 180) & ~((df["lat"] == 0) & (df["long"] == 0))

print("Dropping data centers:", data_centers.loc[~valid_coordinates(data_centers), "facility_name"].tolist())
print("Dropping crypto sites:", crypto.loc[~valid_coordinates(crypto), "Name"].tolist())

data_centers = data_centers[valid_coordinates(data_centers)].reset_index(drop=True)
crypto = crypto[valid_coordinates(crypto)].reset_index(drop=True)

Dropping data centers: []
Dropping crypto sites: ['Cipher "Chief Facility"']


##### Distance between every crypto site and every data center

`geodesic` from the `geopy` package takes two `(lat, long)` points and returns the distance between them along the Earth's surface.

We loop over every crypto site and every data center, which is about 500,000 pairs, so this cell takes around 30 seconds.

The result is a table with one row per crypto site and one column per data center.

In [205]:
crypto_points = list(zip(crypto["lat"], crypto["long"]))
dc_points = list(zip(data_centers["lat"], data_centers["long"]))

distances = []
for crypto_point in crypto_points:
    row = []
    for dc_point in dc_points:
        row.append(geodesic(crypto_point, dc_point).miles)
    distances.append(row)

# turn the list of rows into a table numpy can search
distances = np.array(distances)

print("Distance table (crypto sites, data centers):", distances.shape)

Distance table (crypto sites, data centers): (301, 1708)


##### Nearest data center for each crypto site

Before picking a cutoff, look at how far each crypto site is from its closest data center.

In [206]:
nearest_dc = distances.argmin(axis=1)

crypto["nearest_dc_miles"] = distances.min(axis=1)
crypto["nearest_dc_name"] = data_centers["facility_name"].values[nearest_dc]
crypto["nearest_dc_operator"] = data_centers["operator_name"].values[nearest_dc]

for miles in [0.1, 0.25, 0.5, 1, 2, 5]:
    # keep only the crypto sites that are close enough, then count them
    nearby = crypto[crypto["nearest_dc_miles"] <= miles]
    print(f"Crypto sites with a data center within {miles} mi: {len(nearby)}")

Crypto sites with a data center within 0.1 mi: 23
Crypto sites with a data center within 0.25 mi: 32
Crypto sites with a data center within 0.5 mi: 41
Crypto sites with a data center within 1 mi: 48
Crypto sites with a data center within 2 mi: 58
Crypto sites with a data center within 5 mi: 75


Most crypto sites have no data center nearby. Many crypto mines are simply not in the data center list.

The table below puts the names side by side so we can check whether close pairs are really the same facility.

In [207]:
columns = [
    "Name", "Company", "Precise Location?",
    "nearest_dc_name", "nearest_dc_operator", "nearest_dc_miles"
]

crypto.loc[crypto["nearest_dc_miles"] <= 2, columns].sort_values("nearest_dc_miles")

,Name,Company,Precise Location?,nearest_dc_name,nearest_dc_operator,nearest_dc_miles
55,Core Scientific Whitfield County facility (sel...,CleanSpark,True,Core Scientific Dalton 1 & 2 Data Center Campus,Core Scientific,0.000000
98,Block Mining McCracken County facility,Riot,True,Riot 1,Riot,0.000000
173,Greenidge Generation Yates County facility,Greenidge Generation LLC,True,Vulcan Infrastructure and Power Inc (formerly ...,NaN,0.000000
178,Digipower X Fortistar Niagara County facility,Digipower X,True,North Tonawanda Data Center,NaN,0.000000
194,Bitfarm (Stronghold) Panther Creek Plant,Bitfarm,True,Keel Infrastructure Panther Creek,Keel Infrastructure,0.000000
195,Bitfarm (Stronghold) Scrubgrass Plant,Bitfarm,True,Keel Infrastructure Scrubgrass,Keel Infrastructure,0.000000
202,BitFarms Mercer County facility,BitFarms,True,Sharon Data Center,NaN,0.000061
260,Riot Platforms,Riot Platforms,True,Riot Platforms,NaN,0.000083
154,Atlas Power Williams County facility,FX Solutions > Atlas Power,True,Atlas Power,NaN,0.000139
103,Riot Platforms greenfield site,Riot,False,Riot 2,Riot,0.000171


##### Source 1: the crypto dataset

We label from the data center side: for each data center, find the closest crypto site. If it is within `MAX_MILES`, `crypto_from_distance` is True.

Labeling this way handles two things:
- The crypto file repeats some sites (same name or same coordinates), and several crypto rows can point to one data center.
- Every data center gets exactly one label.

`0.5` miles is a starting point. Some sites under 2 miles are clearly different facilities, for example CleanSpark (Clinton) and Amazon Clinton Data Center.

We also keep the exact address matches from above. Two of them are real matches that are a little more than 0.5 miles apart (Helios Data Center and Riot Platforms Rockdale).

In [208]:
MAX_MILES = 0.5

nearest_crypto = distances.argmin(axis=0)

data_centers["crypto_miles"] = distances.min(axis=0)
data_centers["crypto_name"] = crypto["Name"].values[nearest_crypto]
data_centers["crypto_company"] = crypto["Company"].values[nearest_crypto]
data_centers["crypto_status"] = crypto["Status"].values[nearest_crypto]

# Check 1: a crypto site is within MAX_MILES of the data center
data_centers["crypto_from_distance"] = data_centers["crypto_miles"] <= MAX_MILES

# Check 2: the data center matched a crypto site by exact address
address_match_ids = list(matches["facility_id"])
data_centers["crypto_from_address"] = data_centers["facility_id"].isin(address_match_ids)

print("Crypto from distance:", data_centers["crypto_from_distance"].sum())
print("Crypto from address: ", data_centers["crypto_from_address"].sum())

Crypto from distance: 41
Crypto from address:  19


##### Source 2: the data centers dataset

We search `purpose` and `facility_name` for words that mean crypto mining. We do not search for "crypto" everywhere, because it would also match "NSA Texas Cryptology Center".

In [209]:
# "^crypto" means the text starts with "crypto"
crypto_words = "bitcoin|crypto mining|crypto mine|cryptomine|cryptocurrency|^crypto"

purpose_says_crypto = data_centers["purpose"].str.contains(crypto_words, case=False, na=False)
name_says_crypto = data_centers["facility_name"].str.contains(crypto_words, case=False, na=False)

# Check 3: the data center says it is crypto
data_centers["crypto_from_dc_dataset"] = purpose_says_crypto | name_says_crypto

print("Crypto from purpose or name:", data_centers["crypto_from_dc_dataset"].sum())

Crypto from purpose or name: 21


##### Final label

A data center gets `is_crypto = 1` if any of the three checks is True.

In [210]:
matched_crypto_site = data_centers["crypto_from_distance"] | data_centers["crypto_from_address"]

data_centers["is_crypto"] = (
    matched_crypto_site | data_centers["crypto_from_dc_dataset"]
).astype(int)

# only keep crypto site details for data centers that matched a crypto site
data_centers.loc[~matched_crypto_site, ["crypto_name", "crypto_company", "crypto_status"]] = np.nan

print("Data centers labeled crypto:", data_centers["is_crypto"].sum(), "of", len(data_centers))
print("Share labeled crypto:", round(data_centers["is_crypto"].mean() * 100, 1), "%")

Data centers labeled crypto: 59 of 1708
Share labeled crypto: 3.5 %


In [211]:
columns = [
    "facility_name", "operator_name", "state", "purpose",
    "crypto_from_distance", "crypto_from_address", "crypto_from_dc_dataset",
    "crypto_name", "crypto_miles"
]

data_centers.loc[data_centers["is_crypto"] == 1, columns].sort_values("crypto_miles")

,facility_name,operator_name,state,purpose,crypto_from_distance,crypto_from_address,crypto_from_dc_dataset,crypto_name,crypto_miles
216,Core Scientific Dalton 1 & 2 Data Center Campus,Core Scientific,GA,NaN,True,True,False,Core Scientific Whitfield County facility (sel...,0.000000
451,Riot 1,Riot,KY,NaN,True,True,False,Block Mining McCracken County facility,0.000000
683,Vulcan Infrastructure and Power Inc (formerly ...,NaN,NY,Bitcoin transitioning to AI,True,True,True,Greenidge Generation Yates County facility,0.000000
701,North Tonawanda Data Center,NaN,NY,NaN,True,True,False,Digipower X Fortistar Niagara County facility,0.000000
854,Keel Infrastructure Scrubgrass,Keel Infrastructure,PA,NaN,True,True,False,Bitfarm (Stronghold) Scrubgrass Plant,0.000000
868,Keel Infrastructure Panther Creek,Keel Infrastructure,PA,AI,True,True,False,Bitfarm (Stronghold) Panther Creek Plant,0.000000
902,Sharon Data Center,NaN,PA,NaN,True,True,False,BitFarms Mercer County facility,0.000061
979,Riot Platforms,NaN,TX,NaN,True,False,False,Riot Platforms,0.000083
619,Atlas Power,NaN,ND,Crypto,True,False,True,Atlas Power Williams County facility,0.000139
452,Riot 2,Riot,KY,NaN,True,True,False,Riot Platforms greenfield site,0.000171


##### How much does the cutoff matter?

The number of data centers matched by distance depends on `MAX_MILES`. This helps us choose it. (This table only counts the distance check, not the address or `purpose`/`facility_name` checks.)

In [212]:
rows = []
for miles in [0.1, 0.25, 0.5, 1, 2]:
    labeled_dcs = data_centers[data_centers["crypto_miles"] <= miles]
    matched_crypto = crypto[crypto["nearest_dc_miles"] <= miles]

    rows.append({
        "max_miles": miles,
        "data_centers_labeled": len(labeled_dcs),
        "crypto_sites_matched": len(matched_crypto),
    })

pd.DataFrame(rows)

,max_miles,data_centers_labeled,crypto_sites_matched
0,0.10,23,23
1,0.25,32,32
2,0.50,41,41
3,1.00,51,48
4,2.00,65,58


### EIA-860 Plant Data

The first row of each EIA-860 Excel file is a title, so the column names are on the second row (`header=1`).

In [213]:
# Added header=1 to skip the first row

plants = pd.read_excel(DATA_DIR / "eia860_2025" / "2___Plant_Y2025.xlsx", header=1)
print("Rows and columns:", plants.shape)
plants.sample(3)

Rows and columns: (17349, 42)


,Utility ID,Utility Name,Plant Code,Plant Name,Street Address,City,State,Zip,County,Latitude,Longitude,NERC Region,Balancing Authority Code,Balancing Authority Name,Name of Water Source,Primary Purpose (NAICS Code),Regulatory Status,Sector,Sector Name,FERC Cogeneration Status,FERC Cogeneration Docket Number,FERC Small Power Producer Status,FERC Small Power Producer Docket Number,FERC Exempt Wholesale Generator Status,FERC Exempt Wholesale Generator Docket Number,Ash Impoundment?,Ash Impoundment Lined?,Ash Impoundment Status,Transmission or Distribution System Owner,Transmission or Distribution System Owner ID,Transmission or Distribution System Owner State,Grid Voltage (kV),Grid Voltage 2 (kV),Grid Voltage 3 (kV),Energy Storage,Natural Gas LDC Name,Natural Gas Pipeline Name 1,Natural Gas Pipeline Name 2,Natural Gas Pipeline Name 3,Pipeline Notes,Natural Gas Storage,Liquefied Natural Gas Storage
10276,59496,ALLETE Clean Energy,62938,Glen Ullin Energy Center,6043 30th Street SW,Glen Ullin,ND,58631,Morton,46.964077,-101.816495,MRO,MISO,Midcontinent Independent Transmission System O...,NaN,22,NR,2,IPP Non-CHP,N,NaN,N,NaN,Y,19-95-000,NaN,NaN,NaN,"ALLETE, Inc.",12647,MN,230,,,N,NaN,NaN,NaN,NaN,NaN,N,N
7068,17718,Southwestern Public Service Co,59247,Hale Community Wind Farm,TBD,Plainview,TX,79072,Hale,33.927778,-101.742222,MRO,SWPP,Southwest Power Pool,NaN,22,RE,1,Electric Utility,N,NaN,N,NaN,N,NaN,N,X,NaN,Southwestern Public Service Co,17718,TX,240,,,N,NaN,NaN,NaN,NaN,NaN,X,X
15420,66852,"PVS 2, LLC",68487,Pleasant Valley Solar 2 Project,15904 E Monroe Ave,Boise,ID,83716,Ada,43.327775,-115.988221,WECC,IPCO,Idaho Power Company,NaN,22,NR,2,IPP Non-CHP,N,NaN,N,NaN,N,NaN,N,NaN,NaN,Idaho Power Co,9191,ID,230,,,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [214]:
plants.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Utility ID,17349.0,NaN,NaN,NaN,47106.087844,23529.704706,7.0,18642.0,60301.0,64545.0,68045.0
Utility Name,17349,6935,"Altus Power America Management, LLC",247,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Plant Code,17349.0,NaN,NaN,NaN,52099.079601,22868.185984,1.0,55701.0,61103.0,65921.0,70452.0
Plant Name,17349,17345,Viola,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Street Address,17229,15510,TBD,789,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,17314,6015,Lancaster,78,NaN,NaN,NaN,NaN,NaN,NaN,NaN
State,17349,51,CA,2250,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Zip,17349.0,8431.0,93501.0,64.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
County,17326,1542,Kern,304,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Latitude,17349,16666,,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### EIA-860 Generator Data

The generator file has three sheets: `Operable`, `Proposed`, and `Retired and Canceled`. For now we only look at `Operable`.

In [215]:
generators = pd.read_excel(
    DATA_DIR / "eia860_2025" / "3_1_Generator_Y2025.xlsx",
    sheet_name="Operable",
    header=1
)
print("Rows and columns:", generators.shape)
generators.sample(3)

Rows and columns: (27919, 73)


,Utility ID,Utility Name,Plant Code,Plant Name,State,County,Generator ID,Technology,Prime Mover,Unit Code,Ownership,Duct Burners,Can Bypass Heat Recovery Steam Generator?,RTO/ISO LMP Node Designation,RTO/ISO Location Designation for Reporting Wholesale Sales Data to FERC,Nameplate Capacity (MW),Nameplate Power Factor,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Uprate or Derate Completed During Year,Month Uprate or Derate Completed,Year Uprate or Derate Completed,Status,Synchronized to Transmission Grid,Operating Month,Operating Year,Planned Retirement Month,Planned Retirement Year,Associated with Combined Heat and Power System,Sector Name,Sector,Topping or Bottoming,Energy Source 1,Energy Source 2,Energy Source 3,Energy Source 4,Energy Source 5,Energy Source 6,Startup Source 1,Startup Source 2,Startup Source 3,Startup Source 4,Solid Fuel Gasification System?,Carbon Capture Technology?,Turbines or Hydrokinetic Buoys,Time from Cold Shutdown to Full Load,Fluidized Bed Technology?,Pulverized Coal Technology?,Stoker Technology?,Other Combustion Technology?,Subcritical Technology?,Supercritical Technology?,Ultrasupercritical Technology?,Planned Net Summer Capacity Uprate (MW),Planned Net Winter Capacity Uprate (MW),Planned Uprate Month,Planned Uprate Year,Planned Net Summer Capacity Derate (MW),Planned Net Winter Capacity Derate (MW),Planned Derate Month,Planned Derate Year,Planned New Prime Mover,Planned Energy Source 1,Planned New Nameplate Capacity (MW),Planned Repower Month,Planned Repower Year,Other Planned Modifications?,Other Modifications Month,Other Modifications Year,Multiple Fuels?,Cofire Fuels?,Switch Between Oil and Natural Gas?
27097,64449,"Texas Microgrid, LLC",68736.0,HEBTEMDistCenter2A,TX,Bell,HT2A3,Natural Gas Internal Combustion Engine,IC,NaN,S,X,X,NaN,NaN,0.4,0.8,0.4,0.4,0,N,,,OP,X,11.0,2024.0,,,N,IPP Non-CHP,2.0,X,NG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,N,,10M,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,,,,,,,NaN,NaN,,,,NaN,,,N,NaN,NaN
17716,59541,Carroll County Energy LLC,59773.0,Carroll County Energy,OH,Carroll,CGT2,Natural Gas Fired Combined Cycle,CT,CCE1,S,X,N,1292915936,1292915936,235.5,0.85,213.3,228.3,,N,,,OP,X,12.0,2017.0,,,N,IPP Non-CHP,2.0,X,NG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,,12H,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,,,,,,,NaN,NaN,,,,NaN,,,N,N,N
26870,66793,Rochester MA 2 LLC,68438.0,Rochester MA 2,MA,Plymouth,12998,Solar Photovoltaic,PV,NaN,S,X,X,NaN,NaN,1.0,,1,1,,N,,,OP,X,12.0,2018.0,,,N,IPP Non-CHP,2.0,X,SUN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,,,,,,,NaN,NaN,,,,NaN,,,NaN,NaN,NaN


In [216]:
generators.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Utility ID,27919.0,5853.0,55938.0,329.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Utility Name,27918,5842,Industrial Power Generating Company LLC,329,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Plant Code,27918.0,NaN,NaN,NaN,42375.107386,26782.287436,1.0,7146.0,56794.5,62643.75,70438.0
Plant Name,27918,14325,Edison Sault,73,NaN,NaN,NaN,NaN,NaN,NaN,NaN
State,27918,51,CA,3161,NaN,NaN,NaN,NaN,NaN,NaN,NaN
County,27918,1479,Los Angeles,424,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Generator ID,27917,10890,1,2923,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Technology,27918,27,Solar Photovoltaic,7999,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Prime Mover,27918,18,PV,7999,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Unit Code,2180,310,CC1,770,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Export data prior to failed cells

In [217]:
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

data_centers.to_csv(PROCESSED_DIR / "fractracker_data_centers_clean.csv", index=False)
print("Saved: fractracker_data_centers_clean.csv with", data_centers.shape[0], "rows and", data_centers.shape[1], "columns")

Saved: fractracker_data_centers_clean.csv with 1708 rows and 85 columns


EIA-860 Defining Energy Source Features

In [218]:
energy_sources = pd.read_csv(
    DATA_DIR / "eia860_2025" / "eia_860_energy_source_codes.csv",
)
energy_sources.rename(columns={'Energy Source Code': 'Energy Source 1'}, inplace=True)
energy_sources.sample(3)

FileNotFoundError: [Errno 2] No such file or directory: 'data\\raw\\eia860_2025\\eia_860_energy_source_codes.csv'

In [ ]:
generators['Energy Source 1'].unique()
generator_features = generators[['State','County','Energy Source 1','Technology','Time from Cold Shutdown to Full Load','Summer Capacity (MW)','Winter Capacity (MW)']].copy()
# Dropping description to create categorical feature from Energy Source
generator_features_full = pd.merge(generator_features, energy_sources, on='Energy Source 1', how='left').drop('Description', axis=1)
generator_features_full.rename(columns={'Category':'Energy Category','Time from Cold Shutdown to Full Load':'Time to Full Load'}, inplace=True)
# Clean Time to Full Load from string to numerical value
generator_features_full.sample(15)

Time to Full Load in Minutes Clean Up

In [ ]:
def time_in_min(stringval):
    val = str(stringval).lower().strip()
    # Match digits versus text
    regex = re.match(r"(\d+)(\s*[a-z]+)", val)
    if not regex:
        return None

    if regex.group(2) in ['m']:
        return int(regex.group(1))
    elif regex.group(2) in ['h']:
        return int(regex.group(1))*60
    else:
        return None

In [ ]:
generator_features_full['Time to Full Load'] = generator_features_full['Time to Full Load'].apply(time_in_min)


NameError: name 'generator_features_full' is not defined

In [ ]:
generator_features_full.describe()
generator_features_full['Time to Full Load'] = generator_features_full['Time to Full Load'].fillna(generator_features_full['Time to Full Load'].mean().round(1))

One-Hot Encoding for Energy

In [ ]:
one_hot_energy = pd.get_dummies(generator_features_full['Energy Source 1'])
generator_features_full = generator_features_full.join(one_hot_energy)
generator_features_full.sample(5)

### EIA-861M Retail Sales of Electricity


In [ ]:
# Added header=2 to skip the first two rows of the Excel sheet

sales = pd.read_excel(DATA_DIR / "eia861m_sales_revenue.xlsx", sheet_name="Monthly-States", header=2)
print("Rows and columns:", sales.shape)
sales.sample(3)

In [ ]:
sales.describe(include="all").T

In [ ]:
sales.info()

In [ ]:
sales['Year'] = sales['Year'].astype(str)

In [ ]:
# Looking at 2025 only and only .1 since that is Commercial
# EIA classifies data centers as commercial customers
sales_2025 = sales[sales['Year']=='2025'].copy()
sales_2025 = sales_2025[['Year','Month','State','Thousand Dollars.1','Megawatthours.1','Count.1','Cents/kWh.1']]
sales_2025.sample(5)

In [ ]:
# Group by State, Summer/Winter AVG
# Summer/Winter to match generator dataset capacity values
sales_2025['Season'] = sales_2025['Month'].apply(
    lambda x: 'summer' if 4 <= x <= 9 else 'winter'
)
grouped_sales = sales_2025.groupby(by=['State','Season'], as_index=False).mean(numeric_only=True).sort_values(by='State', ascending=True).drop(columns=['Month'])
grouped_sales.head(10)

### EIA-930 Balance Data

In [ ]:
# This is a subset of the EIA 930 Balance dataset for the second half of 2025

balance = pd.read_csv(DATA_DIR / "eia930" / "EIA930_BALANCE_2025_Jul_Dec.csv")
print("Rows and columns:", balance.shape)
balance.sample(3)

In [ ]:
balance.describe(include="all").T

### EIA-930 Subregion Data

In [ ]:
subregion = pd.read_csv(DATA_DIR / "eia930" / "EIA930_SUBREGION_2025_Jul_Dec.csv")
print("Rows and columns:", subregion.shape)
subregion.sample(3)

In [ ]:
subregion.describe(include="all").T

## Save cleaned data

Save the cleaned data centers, including the `is_crypto` label, to `data/processed/`. This file is the input to `03-supervised-learning.ipynb`.

In [ ]:
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

data_centers.to_csv(PROCESSED_DIR / "fractracker_data_centers_clean.csv", index=False)
print("Saved: fractracker_data_centers_clean.csv with", data_centers.shape[0], "rows and", data_centers.shape[1], "columns")